In [1]:
import cloudscraper

link = "https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pta-surabaya/kategori/perdata-1.html"
# link = "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0d5755564db00be8a313030373136.html"
scraper = cloudscraper.create_scraper()
response = scraper.get(link)
print(response.text)

<!DOCTYPE html>
<html>
  	<head>
		<script src="https://ajax.googleapis.com/ajax/libs/jquery/3.5.1/jquery.min.js"></script>
		<script src="https://ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js"></script>
		<style type="text/css">@font-face {font-family:Source Sans Pro;font-style:normal;font-weight:300;src:url(/cf-fonts/s/source-sans-pro/5.0.11/greek/300/normal.woff2);unicode-range:U+0370-03FF;font-display:swap;}@font-face {font-family:Source Sans Pro;font-style:normal;font-weight:300;src:url(/cf-fonts/s/source-sans-pro/5.0.11/cyrillic/300/normal.woff2);unicode-range:U+0301,U+0400-045F,U+0490-0491,U+04B0-04B1,U+2116;font-display:swap;}@font-face {font-family:Source Sans Pro;font-style:normal;font-weight:300;src:url(/cf-fonts/s/source-sans-pro/5.0.11/cyrillic-ext/300/normal.woff2);unicode-range:U+0460-052F,U+1C80-1C88,U+20B4,U+2DE0-2DFF,U+A640-A69F,U+FE2E-FE2F;font-display:swap;}@font-face {font-family:Source Sans Pro;font-style:normal;font-weight:300;src:url(/cf-fonts/s

In [2]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import os
import time

scraper = cloudscraper.create_scraper()

MAX_RETRIES = 5
RETRY_DELAY = 2  # Initial delay in seconds

def retry_request(func):
    """Decorator to retry a function up to MAX_RETRIES times."""
    def wrapper(*args, **kwargs):
        last_exception = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                result = func(*args, **kwargs)
                if result is not None:
                    return result
                # If result is None, treat as failure and retry
                if attempt < MAX_RETRIES:
                    delay = RETRY_DELAY * (2 ** (attempt - 1))  # Exponential backoff
                    print(f"   ⏳ Attempt {attempt} returned None, retrying in {delay}s...")
                    time.sleep(delay)
            except Exception as e:
                last_exception = e
                if attempt < MAX_RETRIES:
                    delay = RETRY_DELAY * (2 ** (attempt - 1))  # Exponential backoff
                    print(f"   ⚠️ Attempt {attempt} failed: {e}. Retrying in {delay}s...")
                    time.sleep(delay)
                else:
                    print(f"   ❌ All {MAX_RETRIES} attempts failed. Last error: {e}")
        return None
    return wrapper

def sanitize_filename(filename):
    """Sanitize the filename by removing/replacing invalid characters."""
    sanitized = filename.replace('/', '_').replace('\\', '_')
    sanitized = re.sub(r'[<>:"|?*]', '', sanitized)
    sanitized = re.sub(r'[\s_]+', '_', sanitized)
    sanitized = sanitized.strip('_')
    if not sanitized.endswith('.pdf'):
        sanitized += '.pdf'
    return sanitized

@retry_request
def fetch_page(url):
    """Fetch a page with retry logic."""
    response = scraper.get(url, timeout=30)
    if response.status_code == 200:
        return response
    raise Exception(f"HTTP {response.status_code}")

def get_pdf_from_putusan_page(url):
    """Fetch the putusan detail page and extract PDF download link."""
    response = fetch_page(url)
    if not response:
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    pdf_link = soup.find('a', href=lambda x: x and '/pdf/' in x)
    
    if pdf_link:
        return pdf_link.get('href')
    return None

def download_pdf(pdf_url, filename, output_dir='downloads'):
    """Download PDF and save with the given filename."""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    
    # Check if file already exists
    if os.path.exists(filepath):
        print(f"   ⏭️ Skipped (already exists): {filename}")
        return True
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pdf_response = scraper.get(pdf_url, timeout=60)
            if pdf_response.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(pdf_response.content)
                print(f"   ✅ Downloaded: {filename}")
                return True
            else:
                raise Exception(f"HTTP {pdf_response.status_code}")
        except Exception as e:
            if attempt < MAX_RETRIES:
                delay = RETRY_DELAY * (2 ** (attempt - 1))
                print(f"   ⚠️ Download attempt {attempt} failed: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            else:
                print(f"   ❌ Failed to download after {MAX_RETRIES} attempts: {filename}")
                return False
    return False

def parse_putusan_html(html_content):
    """Parse HTML and extract putusan entries."""
    soup = BeautifulSoup(html_content, 'html.parser')
    extracted_data = []

    main_container = soup.find('div', id='tabs-1')
    if not main_container:
        print("Container tabs-1 tidak ditemukan.")
        return []

    entries = main_container.find_all('div', class_='spost')

    for entry in entries:
        item = {}
        
        title_tag = entry.select_one('strong > a')
        
        if title_tag:
            item['title'] = title_tag.get_text(strip=True)
            item['url'] = title_tag['href']
        else:
            for strong in entry.find_all('strong'):
                if strong.find('a'):
                    item['title'] = strong.find('a').get_text(strip=True)
                    item['url'] = strong.find('a')['href']
                    break
            else:
                item['title'] = "Judul Tidak Ditemukan"
                item['url'] = "#"

        small_divs = entry.find_all('div', class_='small')
        for div in small_divs:
            text = div.get_text(strip=True)
            if "Register" in text:
                item['dates'] = " ".join(text.split())
                break
        
        all_divs = entry.find_all('div')
        for div in all_divs:
            text = div.get_text(strip=True)
            if text.startswith("Tanggal") and "Register" not in text:
                item['summary'] = text
                break
        
        extracted_data.append(item)

    return extracted_data

def download_putusan_pdfs(list_page_url, output_dir='downloads'):
    """Main function to download all PDFs from a putusan list page."""
    print(f"📄 Fetching list page: {list_page_url}")
    
    response = fetch_page(list_page_url)
    if not response:
        print("❌ Failed to fetch list page after all retries")
        return
    
    results = parse_putusan_html(response.content)
    print(f"📋 Found {len(results)} putusan entries\n")
    
    success_count = 0
    fail_count = 0
    
    for i, data in enumerate(results, 1):
        title = data.get('title', 'unknown')
        url = data.get('url', '#')
        
        print(f"[{i}/{len(results)}] Processing: {title}")
        
        if url == '#':
            print("   ⚠️ Skipping - No valid URL")
            fail_count += 1
            continue
        
        pdf_url = get_pdf_from_putusan_page(url)
        
        if pdf_url:
            filename = sanitize_filename(title)
            if download_pdf(pdf_url, filename, output_dir):
                success_count += 1
            else:
                fail_count += 1
        else:
            print(f"   ⚠️ No PDF link found")
            fail_count += 1
        
        print("-" * 60)
    
    print(f"\n📊 Summary: {success_count} downloaded, {fail_count} failed")

In [ ]:
for i in range(2, 5):
    list_url = f"https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-denpasar/kategori/perdata-1/page/{i}.html"
    download_putusan_pdfs(list_url, output_dir='downloads')

📄 Fetching list page: https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-denpasar/kategori/perdata-1/page/2.html
📋 Found 20 putusan entries

[1/20] Processing: Putusan PN DENPASAR Nomor 587/Pdt.G/2025/PN Dps
   ⚠️ Download attempt 1 failed: HTTP 429. Retrying in 2s...
   ⚠️ Download attempt 2 failed: HTTP 429. Retrying in 4s...
   ⚠️ Download attempt 3 failed: HTTP 429. Retrying in 8s...
   ✅ Downloaded: Putusan_PN_DENPASAR_Nomor_587_Pdt.G_2025_PN_Dps.pdf
------------------------------------------------------------
[2/20] Processing: Putusan PN DENPASAR Nomor 339/Pdt.G/2025/PN Dps
   ✅ Downloaded: Putusan_PN_DENPASAR_Nomor_339_Pdt.G_2025_PN_Dps.pdf
------------------------------------------------------------
[3/20] Processing: Putusan PN DENPASAR Nomor 812/Pdt.G/2025/PN Dps
   ✅ Downloaded: Putusan_PN_DENPASAR_Nomor_812_Pdt.G_2025_PN_Dps.pdf
------------------------------------------------------------
[4/20] Processing: Putusan PN DENPASAR Nomor 880/Pdt.G/2025/PN Dps
  

In [6]:
link = "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0d5755564db00be8a313030373136.html"
scraper = cloudscraper.create_scraper()
response = scraper.get(link)
soup = BeautifulSoup(response.text, 'html.parser')
# Find the PDF download link - looking for links containing '/pdf/' in href
pdf_links = soup.find_all('a', href=lambda x: x and '/pdf/' in x)
for pdf_link in pdf_links:
    pdf_url = pdf_link.get('href')
    pdf_name = pdf_link.text.strip()
    print(f"PDF Name: {pdf_name}")
    print(f"PDF URL: {pdf_url}")
    if pdf_link:
        pdf_response = scraper.get(pdf_url)
        with open('548_Pdt.G_2025_PTA_Sby.pdf', 'wb') as f:
            f.write(pdf_response.content)
        print("PDF downloaded successfully!")

PDF Name: 548/Pdt.G/2025/PTA_Sby.pdf
PDF URL: https://putusan3.mahkamahagung.go.id/direktori/download_file/f0f4d60a046955ca9fa6b70b622e4769/pdf/zaf0d5755564db00be8a313030373136
PDF downloaded successfully!


# Mahkamah Agung Court Decision Scraper

This notebook scrapes court decision PDFs from putusan3.mahkamahagung.go.id.

**Source URLs pattern:**
- Page 1: `https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-pati/kategori/perdata-1.html`
- Page N: `https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-pati/kategori/perdata-1/page/{N}.html`

In [ ]:
# Import the scraper module
from judgement_scraper import scrape_judgements, JudgementScraper, ScraperConfig

In [ ]:
# Scrape court decisions - specify how many pages to scrape
# Each page contains ~10 decisions
#
# Parameters:
#   num_pages: Number of listing pages to scrape
#   court: Court identifier (default: 'pn-pati')
#   category: Category identifier (default: 'perdata-1')
#   output_dir: Output directory for PDFs (default: 'data/judgement')
#   page_delay: Delay between page requests in seconds (default: 2.0)
#   detail_delay: Delay between detail/download requests in seconds (default: 1.5)

stats = scrape_judgements(
    num_pages=1,  # Change this to scrape more pages
    output_dir="../data/judgement"  # Relative path from notebook
)

print(f"\nScrape Summary:")
print(f"  Pages scraped: {stats['pages_scraped']}")
print(f"  Decisions found: {stats['decisions_found']}")
print(f"  PDFs downloaded: {stats['pdfs_downloaded']}")
print(f"  PDFs failed: {stats['pdfs_failed']}")

In [ ]:
# Advanced usage: Custom configuration
# Uncomment below to use custom settings

# config = ScraperConfig(
#     court="pn-pati",
#     category="perdata-1", 
#     output_dir="../data/judgement",
#     page_delay=3.0,  # Longer delay for slow connections
#     detail_delay=2.0,
#     max_retries=5
# )
# 
# scraper = JudgementScraper(config)
# stats = scraper.scrape_pages(num_pages=3)

In [ ]:
# List downloaded PDFs
from pathlib import Path

pdf_dir = Path("../data/judgement")
pdfs = list(pdf_dir.glob("*.pdf"))

print(f"Total PDFs in output directory: {len(pdfs)}")
for pdf in pdfs[:10]:  # Show first 10
    print(f"  - {pdf.name}")

if len(pdfs) > 10:
    print(f"  ... and {len(pdfs) - 10} more")